# Linear regression — Boomerang & Zig-Zag (sticky and non-sticky) vs NUTS


## 0. Config

In [ ]:
# --- Reproducibility ---
SEED = 0

# --- Data ---
N, D            = 200, 9
SIGNAL_IDX      = [0, 3, 7]
SIGNAL_VALS     = [1.5, -2.0, 0.8]
INTERCEPT_TRUE  = 0.5
NOISE_STD       = 0.3

# --- Prior / likelihood (must match what NUTS uses) ---
PRIOR_STD            = 1.0
INTERCEPT_PRIOR_STD  = 10.0
LIK_NOISE_STD        = 0.5

# --- Sampler budgets ---
N_SKEL       = 50_000     # PDMP skeleton points
N_RESAMPLE   = 50_000     # path-resampled draws
BURNIN_FRAC  = 0.5        # fraction of skeleton dropped before resampling
REFRESH_RATE = 1.0
KAPPA_NULL   = 1.0        # stickiness for non-intercept coords
KAPPA_INT    = 1e6        # intercept never sticks

# --- Thinning method (applies to all four PDMP samplers) ---
THINNING     = "brent"   # or "pli"

# --- Zig-Zag specifics ---
T_MAX_ZZ     = 0.1
GAMMA_ZZ     = 0.01

# --- NUTS ---
NUTS_DRAWS    = 2_000
NUTS_TUNE     = 1_000
NUTS_CHAINS   = 2
NUTS_TARGET_ACCEPT = 0.9


In [ ]:
import os, sys, time, warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# If running from notebooks/<dir>/<this>.ipynb, walk up to the repo root.
# Adjust if your layout differs; the import below will fail loudly otherwise.
if 'sazz' not in sys.modules:
    try:
        os.chdir('../..')
    except FileNotFoundError:
        pass

import numpy as np
import torch
import matplotlib.pyplot as plt
import pymc as pm

from sazz.samplers.AutomaticBoomerangSampler import AutomaticBoomerangSampler
from sazz.samplers.StickyAutomaticBoomerangSampler import StickyAutomaticBoomerangSampler
from sazz.samplers.AutomaticZigZagSampler_torch import AutomaticZigZagSampler
from sazz.samplers.StickyAutomaticZigZagSampler_torch import StickyAutomaticZigZagSampler
from sazz.models.make_models import make_linear_regression
from sazz.utils.sampling import (
    resample_boomerang_path, resample_boomerang_path_sticky,
    resample_zigzag_path, resample_zigzag_path_sticky,
)


def set_seed(seed: int):
    """Seed every RNG that downstream code might touch."""
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed(SEED)
print(f"torch={torch.__version__}  numpy={np.__version__}  pymc={pm.__version__}")


## 1. Data

Sparse linear model: 3 true signals among 10 features, plus an intercept.
Features are standardised before fitting.


In [ ]:
set_seed(SEED)
rng = np.random.default_rng(SEED)

X = rng.normal(size=(N, D))
beta_true = np.zeros(D)
beta_true[SIGNAL_IDX] = SIGNAL_VALS
y = X @ beta_true + INTERCEPT_TRUE + NOISE_STD * rng.normal(size=N)

X = (X - X.mean(0)) / X.std(0)

X_t = torch.tensor(X, dtype=torch.float64)
y_t = torch.tensor(y, dtype=torch.float64)

# True coefficient vector: [intercept, beta_1, ..., beta_D]
true_coefs = np.concatenate([[INTERCEPT_TRUE], beta_true])
is_signal  = true_coefs != 0
coef_names = ['intercept'] + [f'β_{i}' for i in range(1, len(true_coefs))]
print(f"N={N}, D={D}, signals={int(is_signal.sum())}/{len(true_coefs)}")


## 2. OLS sanity check


In [ ]:
X_aug      = np.column_stack([np.ones(N), X])
ols_coefs  = np.linalg.solve(X_aug.T @ X_aug, X_aug.T @ y)
resid      = y - X_aug @ ols_coefs
sigma2_hat = (resid ** 2).sum() / (N - X_aug.shape[1])
ols_stds   = np.sqrt(np.diag(sigma2_hat * np.linalg.inv(X_aug.T @ X_aug)))

print(f"{'coef':<10} {'true':>8} {'OLS':>8} {'OLS std':>9}")
for i, (t, m, s) in enumerate(zip(true_coefs, ols_coefs, ols_stds)):
    star = ' *' if t != 0 else ''
    print(f"{coef_names[i]:<10} {t:>8.3f} {m:>8.3f} {s:>9.3f}{star}")


## 3. NUTS reference

PyMC's NUTS targets the same posterior the Boomerang samplers do (same
priors, same likelihood). It plays the role of the gold standard.


In [ ]:
set_seed(SEED)

with pm.Model() as linreg_model:
    intercept = pm.Normal('intercept', mu=0.0, sigma=INTERCEPT_PRIOR_STD)
    betas     = pm.Normal('betas',     mu=0.0, sigma=PRIOR_STD, shape=D)
    mu        = intercept + X @ betas
    pm.Normal('y', mu=mu, sigma=LIK_NOISE_STD, observed=y)

    t0 = time.perf_counter()
    nuts_trace = pm.sample(
        draws=NUTS_DRAWS, tune=NUTS_TUNE, chains=NUTS_CHAINS,
        target_accept=NUTS_TARGET_ACCEPT, progressbar=False, random_seed=SEED,
    )
    t_nuts = time.perf_counter() - t0

# Flatten to [intercept, β_1, ..., β_D]
samples_nuts = np.column_stack([
    nuts_trace.posterior['intercept'].values.reshape(-1),
    nuts_trace.posterior['betas'].values.reshape(-1, D),
])
print(f"NUTS: {samples_nuts.shape[0]} draws  |  wall = {t_nuts:.2f}s")


## 4. Build the four PDMP samplers

Same target, four variants, all using the same thinning method (`THINNING` set above):

| name           | dynamics  | sticky? |
|----------------|-----------|---------|
| `Boom`         | Boomerang | no      |
| `Sticky-Boom`  | Boomerang | yes     |
| `ZZ`           | Zig-Zag   | no      |
| `Sticky-ZZ`    | Zig-Zag   | yes     |


In [ ]:
target = make_linear_regression(
    X_t, y_t,
    prior_std=PRIOR_STD,
    intercept_prior_std=INTERCEPT_PRIOR_STD,
    noise_std=LIK_NOISE_STD,
    diagonal_only=True,
)

kappa = torch.full((target.D,), KAPPA_NULL, dtype=torch.float64)
kappa[0] = KAPPA_INT  # intercept never sticks


def build_sampler(kind: str):
    """
    kind in {'boom', 'sticky_boom', 'zz', 'sticky_zz'}.
    Thinning is read from the global THINNING.
    """
    if kind == 'boom':
        s = AutomaticBoomerangSampler(
            grad_target=target.grad_target, D=target.D,
            t_max=T_MAX_ZZ, refresh_rate=REFRESH_RATE, thinning=THINNING,
        )
        s.preprocess(x_ref=target.x_ref, Sigma_inv=target.Sigma_inv)
    elif kind == 'sticky_boom':
        s = StickyAutomaticBoomerangSampler(
            grad_target=target.grad_target, D=target.D,
            t_max=T_MAX_ZZ, refresh_rate=REFRESH_RATE, thinning=THINNING,
            kappa=kappa,
        )
        s.preprocess(x_ref=target.x_ref, Sigma_inv=target.Sigma_inv)
    elif kind == 'zz':
        s = AutomaticZigZagSampler(
            grad_target=target.grad_target, D=target.D,
            t_max=T_MAX_ZZ, gamma=GAMMA_ZZ, thinning=THINNING
        )
    elif kind == 'sticky_zz':
        s = StickyAutomaticZigZagSampler(
            grad_target=target.grad_target, D=target.D,
            t_max=T_MAX_ZZ, gamma=GAMMA_ZZ, thinning=THINNING,
            kappa=kappa,
        )
    else:
        raise ValueError(f'Unknown sampler kind: {kind}')
    return s


# Display order is preserved through the rest of the notebook.
SAMPLERS = {
    'Boom'       : 'boom',
    'Sticky-Boom': 'sticky_boom',
    'ZigZag'         : 'zz',
    'Sticky-ZigZag'  : 'sticky_zz',
}


## 5. Run all four samplers

Each sampler is reseeded immediately before sampling, so the order in which
they're listed in the dict above does not affect any individual result.


In [ ]:
def run_one(kind: str):
    """Build, sample, resample, time. Returns (samples, wall_seconds, raw)."""
    set_seed(SEED)
    sampler = build_sampler(kind)

    # Same starting point for all samplers, for fair comparison.
    x0 = target.x_ref.clone() + torch.randn(target.D)

    t0 = time.perf_counter()
    res = sampler.sample(N=N_SKEL, x0=x0, diagnostics=True)
    t_sample = time.perf_counter() - t0

    # Dispatch to the appropriate path resampler
    pos = res['positions'].cpu().numpy()
    vel = res['velocities'].cpu().numpy()
    tim = res['times'].cpu().numpy()

    if kind == 'boom':
        samples = resample_boomerang_path(
            pos, vel, tim, target.x_ref.cpu().numpy(),
            N_resample=N_RESAMPLE, burnin_frac=BURNIN_FRAC,
        )
    elif kind == 'sticky_boom':
        samples = resample_boomerang_path_sticky(
            pos, vel, tim, target.x_ref.cpu().numpy(),
            N_resample=N_RESAMPLE, burnin_frac=BURNIN_FRAC,
        )
    elif kind == 'zz':
        samples = resample_zigzag_path(
            pos, vel, tim,
            N_resample=N_RESAMPLE, burnin_frac=BURNIN_FRAC,
        )
    elif kind == 'sticky_zz':
        samples = resample_zigzag_path_sticky(
            pos, vel, tim,
            N_resample=N_RESAMPLE, burnin_frac=BURNIN_FRAC,
        )
    else:
        raise ValueError(f'Unknown sampler kind: {kind}')

    return samples, t_sample, res


results = {}   # name -> dict(samples, wall, raw, kind)
for name, kind in SAMPLERS.items():
    samples, wall, raw = run_one(kind)
    sticky = kind.startswith('sticky')
    results[name] = dict(samples=samples, wall=wall, raw=raw,
                         kind=kind, sticky=sticky)
    print(f'{name:<13}  wall={wall:6.2f}s  resampled={samples.shape[0]}')


## 6. Wall-clock summary

In [ ]:
walls = {'NUTS': t_nuts, **{n: r['wall'] for n, r in results.items()}}

print(f"{'sampler':<14} {'wall (s)':>9}")
print('-' * 24)
for name, w in walls.items():
    print(f"{name:<14} {w:>9.2f}")


## 7. Coefficient summary table


In [ ]:
def summarize(samples):
    return samples.mean(0), samples.std(0), (np.abs(samples) < 1e-8).mean(0)


# Headers
cols = [('NUTS', samples_nuts, False)]
cols += [(name, r['samples'], r['sticky']) for name, r in results.items()]

# Print header
header_parts = [f"{'coef':<10} {'true':>7} {'OLS':>7}"]
for name, _, sticky in cols:
    if sticky:
        header_parts.append(f"{name+' μ':>13} {name+' σ':>11} {'P(=0)':>6}")
    else:
        header_parts.append(f"{name+' μ':>13} {name+' σ':>11}")
header = '  |  '.join(header_parts)
print(header)
print('-' * len(header))

# Precompute summaries
summaries = [summarize(s) for _, s, _ in cols]

for i in range(len(true_coefs)):
    star = ' *' if is_signal[i] else ''
    parts = [f"{coef_names[i]:<10} {true_coefs[i]:>7.3f} {ols_coefs[i]:>7.3f}"]
    for (name, _, sticky), (mu, sd, p0) in zip(cols, summaries):
        if sticky:
            parts.append(f"{mu[i]:>13.3f} {sd[i]:>11.3f} {p0[i]:>6.2f}")
        else:
            parts.append(f"{mu[i]:>13.3f} {sd[i]:>11.3f}")
    print('  |  '.join(parts) + star)


In [ ]:
def summarize(samples):
    return samples.mean(0), samples.std(0), (np.abs(samples) < 1e-8).mean(0)


def fmt(x, digits=3):
    s = f'{x:.{digits}f}'
    s = s.rstrip('0').rstrip('.')
    if s == '-0':
        s = '0'
    return s


def print_latex_table(true_coefs, coef_names, samples_nuts, results):
    cols = [('NUTS', samples_nuts, False)]
    cols += [(name, r['samples'], r['sticky']) for name, r in results.items()]

    summaries = [summarize(s) for _, s, _ in cols]

    print(r'\begin{table}[htbp]')
    print(r'\centering')
    print(r'\small')
    print(r'\setlength{\tabcolsep}{3pt}')

    # One column per method now (sticky info folded into the cell)
    n_cols = 2 + len(cols)
    print(r'\begin{tabular}{' + 'l' + 'r' * (n_cols - 1) + '}')
    print(r'\toprule')

    header = ['coef', 'true'] + [name for name, _, _ in cols]
    print(' & '.join(header) + r' \\')
    print(r'\midrule')

    for i in range(len(true_coefs)):
        coef_label = coef_names[i]
        if 'beta' in coef_label.lower():
            idx = coef_label.split('_')[-1]
            coef_label = rf'$\beta_{{{idx}}}$'

        row = [coef_label, fmt(true_coefs[i], 2)]

        for (_, _, sticky), (mu, sd, p0) in zip(cols, summaries):
            cell = f'{fmt(mu[i])} ({fmt(sd[i])})'
            if sticky:
                cell += f' [{p0[i]:.2f}]'
            row.append(cell)

        line = ' & '.join(row)
        print(line + r' \\')

    print(r'\bottomrule')
    print(r'\end{tabular}')
    print(r'\caption{Parameter estimates across methods. '
          r'Cells show posterior mean (std); sticky methods additionally '
          r'report $[P_0]$, the fraction of draws clamped at zero. '
          r'Signal coefficients marked $^{*}$.}')
    print(r'\end{table}')

In [ ]:
# call site
import io, sys
latex_str = io.StringIO()
old_stdout = sys.stdout
sys.stdout = latex_str
print_latex_table(true_coefs, coef_names, samples_nuts, results)
sys.stdout = old_stdout
print(latex_str.getvalue())

## 8. Calibration plot


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))
idx = np.arange(len(true_coefs))

plot_order = [
    ('NUTS',        samples_nuts,                          'D', 'C2'),
    ('Boom',        results['Boom']['samples'],            'o', 'C0'),
    ('Sticky-Boom', results['Sticky-Boom']['samples'],     's', 'C1'),
    ('ZigZag',          results['ZigZag']['samples'],              'v', 'C9'),
    ('Sticky-ZigZag',   results['Sticky-ZigZag']['samples'],       '^', 'C3'),
]
n_methods = len(plot_order)
offsets = np.linspace(-0.30, 0.30, n_methods)

for (name, s, marker, color), off in zip(plot_order, offsets):
    mu = s.mean(0); sd = s.std(0)
    ax.errorbar(idx + off, mu, yerr=2 * sd, fmt=marker, color=color,
                label=f'{name} (μ ± 2σ)', capsize=2, markersize=4, lw=1)

ax.scatter(idx, true_coefs, marker='x', color='k', s=70, linewidths=2,
           label='true', zorder=5)

for i in np.where(is_signal)[0]:
    ax.axvspan(i - 0.45, i + 0.45, color='gold', alpha=0.12)

ax.axhline(0, color='grey', lw=0.5)
ax.set_xticks(idx)
ax.set_xticklabels(coef_names, rotation=0)
ax.set_ylabel('coefficient value')
ax.set_title(f'Posterior intervals — NUTS reference vs all four PDMP variants ({THINNING})')
ax.legend(loc='best', frameon=False, fontsize=8, ncol=2)
plt.tight_layout()
plt.show()


## 9. Trace plots


In [ ]:
def trace_grid(coord_idx: int, ylabel: str):
    fig, axes = plt.subplots(1, 5, figsize=(16, 2.8), sharey=True)
    panels = [
        ('NUTS',        samples_nuts,                          'C2'),
        ('Boom',        results['Boom']['samples'],            'C0'),
        ('Sticky-Boom', results['Sticky-Boom']['samples'],     'C1'),
        ('ZigZag',          results['ZigZag']['samples'],              'C9'),
        ('Sticky-ZigZag',   results['Sticky-ZigZag']['samples'],       'C3'),
    ]
    for ax, (name, samp, color) in zip(axes, panels):
        ax.plot(samp[:, coord_idx], lw=0.3, color=color)
        ax.axhline(true_coefs[coord_idx], color='k', lw=1, ls='--',
                   label=f'true = {true_coefs[coord_idx]:.2f}')
        ax.set_title(name, fontsize=10)
        ax.set_xlabel('sample index')
        ax.legend(loc='best', fontsize=7, frameon=False)
    axes[0].set_ylabel(ylabel)
    plt.tight_layout()
    plt.show()


j_signal = int(np.argmax(np.abs(true_coefs)))
j_null   = int(np.argmin(np.abs(true_coefs[1:])) + 1)  # skip intercept

trace_grid(j_signal, f'{coef_names[j_signal]} (signal)')
trace_grid(j_null,   f'{coef_names[j_null]} (null)')


## 10. Quick numerical summary vs NUTS

Four metrics per sampler, with NUTS reported alongside as a peer rather
than as the gold standard:

- **RMSE on β (parameter recovery):** distance from posterior mean to the
  true generating coefficients, averaged over all coordinates. Sticky
  trades off here — it shrinks signals slightly (small cost) and zeroes
  out nulls (large benefit). Lower is better.
- **Predictive RMSE (test set):** posterior predictive RMSE on a fresh
  noise-free test set drawn from the true DGP, with predictions
  *marginalised* over the full posterior. Lower is better.
- **σ ratio (signals only):** posterior std on signal coordinates,
  divided by NUTS's. ≈ 1.0 means well-calibrated.
- **P(=0) on nulls** (sticky only): average exclusion probability on null
  coefficients. Higher = more aggressive sparsity.

The test set has the same N as training and is drawn fresh from the same
standardised DGP as Section 1.


In [ ]:
# --- Build a fresh test set from the true DGP ---
test_rng = np.random.default_rng(SEED + 1)
X_test_raw = test_rng.normal(size=(N, D))
X_test = (X_test_raw - X_test_raw.mean(0)) / X_test_raw.std(0)
y_test_clean = X_test @ beta_true + INTERCEPT_TRUE
X_test_aug = np.column_stack([np.ones(N), X_test])


def metric(samples: np.ndarray) -> float:
    """Posterior-predictive RMSE on the noise-free test set."""
    preds = samples @ X_test_aug.T
    pred_mean = preds.mean(axis=0)
    return float(np.sqrt(((pred_mean - y_test_clean) ** 2).mean()))


def beta_rmse(samples: np.ndarray) -> float:
    """RMSE between posterior-mean coefficients and true coefficients."""
    return float(np.sqrt(((samples.mean(0) - true_coefs) ** 2).mean()))


all_samplers = [('NUTS', samples_nuts, False)]
all_samplers += [(n, r['samples'], r['sticky']) for n, r in results.items()]

sd_n = samples_nuts.std(0)
print(f"{'sampler':<14}  {'RMSE on β':>10}  {'Pred. RMSE':>10}  "
      f"{'σ-ratio (sig)':>14}  {'P(=0) on nulls':>16}")
print('-' * 75)
for name, s, sticky in all_samplers:
    sd = s.std(0); p0 = (np.abs(s) < 1e-8).mean(0)
    b_rmse = beta_rmse(s)
    m_val  = metric(s)
    ratio  = (sd[is_signal] / sd_n[is_signal]).mean()
    p0nl   = p0[~is_signal].mean() if sticky else float('nan')
    p0nl_str = f'{p0nl:>16.2f}' if not np.isnan(p0nl) else f'{"—":>16}'
    print(f'{name:<14}  {b_rmse:>10.4f}  {m_val:>10.4f}  '
          f'{ratio:>14.2f}  {p0nl_str}')


In [ ]:
def plot_trajectory_comparison(
    results,
    coord_idx=(1, 2),
    n_show=1000,
    start=1000,
    samplers=("ZigZag", "Boom", "Sticky-ZigZag", "Sticky-Boom"),
    figsize=(12, 6),
):
    i, j = coord_idx
    fig, axes = plt.subplots(2, len(samplers)//2, figsize=figsize, squeeze=False)
    axes = axes.flatten()

    for ax, name in zip(axes, samplers):
        samples = results[name]["samples"]
        seg = samples[start : start + n_show, [i, j]]
        # Connect consecutive samples with a thin line
        ax.plot(seg[:, 0], seg[:, 1], lw=0.5, alpha=0.7, color="C0")
        # Lightly mark the points themselves.
        ax.scatter(seg[:, 0], seg[:, 1], s=1, alpha=0.3, color="C0")
        # Mark start and end so you can see direction of travel.
        ax.scatter(seg[0, 0], seg[0, 1], s=40, color="green",
                   zorder=5, label="start", edgecolor="k", linewidth=0.5)
        ax.scatter(seg[-1, 0], seg[-1, 1], s=40, color="red",
                   zorder=5, label="end", edgecolor="k", linewidth=0.5)

        ax.set_xlabel(rf"$\beta_{{{i}}}$")
        ax.set_ylabel(rf"$\beta_{{{j}}}$")
        ax.set_title(f"{name}")
        ax.set_aspect("equal", adjustable="datalim")
        ax.legend(loc="best", fontsize=8)
        ax.grid(True, alpha=0.3)

    fig.suptitle(
        rf"Trajectory comparison in $(\beta_{{{i}}}, \beta_{{{j}}})$-plane",
        fontsize=12,
    )
    fig.tight_layout()

    return fig, axes


In [ ]:
fig, axes = plot_trajectory_comparison(
    results,
    coord_idx=(1, 2),
    n_show=10_000,        # try 1000–5000; lower = clearer geometry
    start=1000,         # skip a bit of warm-up
)
plt.show()

In [ ]:
import numpy as np
import arviz as az

def ess_continuous(result):
    samples = np.asarray(result["samples"])           # [M, D]
    M, D = samples.shape
    ess = np.empty(D)
    for d in range(D):
        ess[d] = float(az.ess(samples[np.newaxis, :, d]))
    return ess

def ess_active(result, zero_tol=1e-8):
    """
    Per-coordinate ESS computed only on draws where the coordinate is
    active (|beta_i| > zero_tol). Tells you about within-slab mixing for
    sticky samplers. Returns NaN for coordinates with too few active
    draws.
    """
    samples = np.asarray(result["samples"])           # [M, D]
    M, D = samples.shape
    active = np.abs(samples) > zero_tol               # [M, D]

    ess = np.full(D, np.nan)
    for d in range(D):
        sub = samples[active[:, d], d]
        if len(sub) >= 50:
            ess[d] = ess[d] = float(az.ess(sub[np.newaxis, :]))
    return ess

def ess_indicator(result, zero_tol=1e-8):
    """
    Per-coordinate ESS of the binary inclusion indicator
    gamma_i = 1[|beta_i| > zero_tol]. Tells you how well the chain
    mixes the spike-vs-slab decision. Returns M for coordinates that
    are always active or always frozen (constant chain).
    """
    samples = np.asarray(result["samples"])           # [M, D]
    M, D = samples.shape
    indicator = (np.abs(samples) > zero_tol).astype(float)

    ess = np.empty(D)
    for d in range(D):
        col = indicator[:, d]
        if col.std() < 1e-12:
            ess[d] = float(M)
        else:
            ess[d] = float(az.ess(col[np.newaxis, :]))
    return ess



In [ ]:
"""
Print a comparison table of ESS across samplers.
"""

def _final_time(result):
    """Final simulation time from result['raw']['times'][-1], else None."""
    raw = result.get("raw")
    if raw is None:
        return None
    times = raw.get("times")
    if times is None:
        return None
    if hasattr(times, "cpu"):
        times = times.cpu().numpy()
    return float(np.asarray(times)[-1])


def print_ess_table(results, nuts_samples=None, nuts_wall=None, zero_tol=1e-8):
    """
    Print a comparison table of ESS across samplers.

    Parameters
    ----------
    results : dict
        Maps sampler name -> {'samples': [M, D], 'wall': float, 'raw': {...}, ...}.
        Sampler names containing 'sticky' (case-insensitive) get the
        active + indicator treatment; others get standard continuous ESS.
    nuts_samples : array [M, D], optional
        NUTS draws (already flattened across chains). Standard ESS.
    nuts_wall : float, optional
        NUTS wall time in seconds. Used to compute ESS/s.
    """
    rows = []  # (name, ess_label, ess_min, ess_median, ess_per_t_min, time_unit)

    for name, r in results.items():
        is_sticky = "sticky" in name.lower()
        T = _final_time(r)
        wall = r.get("wall")

        if not is_sticky:
            ess = ess_continuous(r)
            rows.append({
                "name": name,
                "metric": "ESS",
                "ess_min": np.min(ess),
                "ess_med": np.median(ess),
                "T": T,
                "wall": wall,
            })
        else:
            ess_a = ess_active(r, zero_tol=zero_tol)
            ess_i = ess_indicator(r, zero_tol=zero_tol)
            rows.append({
                "name": name,
                "metric": "active",
                "ess_min": np.nanmin(ess_a),
                "ess_med": np.nanmedian(ess_a),
                "T": T,
                "wall": wall,
            })
            rows.append({
                "name": name,
                "metric": "indicator",
                "ess_min": np.min(ess_i),
                "ess_med": np.median(ess_i),
                "T": T,
                "wall": wall,
            })

    if nuts_samples is not None:
        import arviz as az
        ns = np.asarray(nuts_samples)
        if ns.ndim == 2:
            # Single chain fallback — will give noisy ESS for NUTS
            ns = ns[np.newaxis]
        # ns is now [chains, draws, D]
        D = ns.shape[-1]
        ess_nuts = np.array([float(az.ess(ns[..., d])) for d in range(D)])
        rows.append({
            "name": "NUTS",
            "metric": "ESS",
            "ess_min": np.min(ess_nuts),
            "ess_med": np.median(ess_nuts),
            "T": None,
            "wall": nuts_wall,
        })

    # --- Render ---
    h1 = f"{'Sampler':<18s}{'Metric':<12s}"
    h2 = f"{'min':>10s}{'median':>10s}"
    h3 = f"{'min/T':>12s}{'median/T':>12s}"
    h4 = f"{'min/s':>12s}{'median/s':>12s}"
    print(h1 + h2 + h3 + h4)
    print("-" * len(h1 + h2 + h3 + h4))

    for row in rows:
        name = row["name"]
        metric = row["metric"]
        emin = row["ess_min"]
        emed = row["ess_med"]
        T = row["T"]
        wall = row["wall"]

        line = f"{name:<18s}{metric:<12s}{emin:>10.0f}{emed:>10.0f}"
        if T:
            line += f"{emin/T:>12.2f}{emed/T:>12.2f}"
        else:
            line += f"{'—':>12s}{'—':>12s}"
        if wall:
            line += f"{emin/wall:>12.2f}{emed/wall:>12.2f}"
        else:
            line += f"{'—':>12s}{'—':>12s}"
        print(line)

In [ ]:
# Before, in your harness:
samples_nuts = np.column_stack([
    nuts_trace.posterior['intercept'].values.reshape(-1),
    nuts_trace.posterior['betas'].values.reshape(-1, D),
])
# This gives [4000, 10] — flattened, loses chain structure.

# Replace with:
intercept_3d = nuts_trace.posterior['intercept'].values[..., None]  # [chains, draws, 1]
betas_3d     = nuts_trace.posterior['betas'].values                  # [chains, draws, D]
samples_nuts = np.concatenate([intercept_3d, betas_3d], axis=-1)     # [chains, draws, D+1]

print_ess_table(results, nuts_samples=samples_nuts, nuts_wall=t_nuts)

In [ ]:
import arviz as az

# Direct ESS from the trace, with proper chain structure
print(az.ess(nuts_trace))